# AI Labor Market Intelligence — تشغيل وتدقيق من Google Drive (إصدار مراجَع V2)

**الهدف:** التحقق من سلامة الملفات، إصلاح خطأ AMP المعروف وأخطاء البحث المعروفة، تدريب AraBERT، ثم تنفيذ تقييم واختبارات قابلة للتتبع.هذا الدفتر لم يُنفّذ بالكامل على Google Colab ضمن إعداد هذه النسخة، ولذلك لا يحتوي على نتائج تدريب مُدّعاة.

**قبل البدء:** ارفع `LMI_PROJECT.zip` إلى جذر My Drive، أو ارفع الملفات الستة `LMI_01_...zip` حتى `LMI_06_...zip` إلى `My Drive/LMI_INPUT`، أو استخدم مشروع `My Drive/LMI_PROJECT` إذا كان موجوداً بالفعل. اختر GPU إن توفر؛ CPU سيعمل لكنه أبطأ بكثير.

**نطاق استكمال التدريب:** لا تتجاوز هذه النسخة نموذجاً إلا إذا أمكن تحميل ملف أوزانه ورأسه وTokenizer فعلياً. أما الانقطاع أثناء الحقبة فلا يمكن استئنافه تلقائياً من منتصفها لأن سكربتات التدريب الأصلية لا تحفظ Optimizer/step checkpoints؛ يُعاد تدريب المهمة غير المكتملة. لا تغيّر نتائج Silver إلى Gold تلقائياً.

## 1) ربط Google Drive وتحضير نسخة عمل محلية جديدة
هذه الخلية تنسخ المشروع من Drive إلى `/content/lmi` كل مرة، لضمان عدم استخدام كود قديم متبقٍ من تشغيل سابق. لا تحذف المشروع المحفوظ في Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, shutil, zipfile, subprocess, json, time, glob
MY = Path('/content/drive/MyDrive')
PROJ = MY / 'LMI_PROJECT'
ROOT = Path('/content/lmi')

def unpack_lmi_archive(archive: Path) -> None:
    with zipfile.ZipFile(archive) as z:
        bad = [n for n in z.namelist() if n.startswith('/') or '..' in Path(n).parts or ('\\' in n)]
        if bad:
            raise ValueError(f'Unsafe archive member(s) in {archive}: {bad[:3]}')
        members = [m for m in z.infolist() if m.filename.startswith('LMI_PROJECT/')]
        if not members:
            raise ValueError(f'Archive {archive} does not contain LMI_PROJECT/')
        z.extractall(MY, members=members)
    print('Extracted:', archive.name)

if not (PROJ / 'src/common.py').is_file():
    whole = MY / 'LMI_PROJECT.zip'
    parts = sorted((MY / 'LMI_INPUT').glob('LMI_*.zip'))
    if whole.is_file():
        unpack_lmi_archive(whole)
    elif len(parts) == 6 and sorted(p.name.split('_')[1] for p in parts) == ['01','02','03','04','05','06']:
        for p in parts:
            unpack_lmi_archive(p)
    else:
        raise FileNotFoundError('Expected MyDrive/LMI_PROJECT, MyDrive/LMI_PROJECT.zip, or six LMI_01..06 ZIPs in MyDrive/LMI_INPUT')
if not (PROJ / 'src/common.py').is_file():
    raise FileNotFoundError('The project was not extracted as MyDrive/LMI_PROJECT/src/common.py')
if ROOT.exists():
    shutil.rmtree(ROOT)  # remove only ephemeral /content/lmi; Drive source remains intact
shutil.copytree(PROJ, ROOT, ignore=shutil.ignore_patterns('models_dl','reports','*.bak'))
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['LMI_FINAL_DIR'] = str(ROOT / 'FINAL_TRAINING_DATASET')
os.environ['LMI_MODELS_DIR'] = str(PROJ / 'models_dl')
os.environ['LMI_REPORTS_DIR'] = str(PROJ / 'reports')
(PROJ / 'models_dl').mkdir(exist_ok=True)
(PROJ / 'reports').mkdir(exist_ok=True)
print('Drive project:', PROJ)
print('Local worktree:', ROOT)
print('Dataset available:', (ROOT/'FINAL_TRAINING_DATASET/unique_opportunities.csv').exists())


## 2) تثبيت مكتبات بإصدارات متوافقة

نستخدم `requirements_dl.txt` الخاص بالمشروع (وفيه `transformers<5`) بدلاً من شرط مفتوح قد يثبّت إصداراً غير متوافق. التثبيت يُوقف الدفتر إذا فشل.

In [ ]:
import subprocess, sys
req = ROOT / 'requirements_dl.txt'
if not req.is_file():
    raise FileNotFoundError(f'Missing {req}')
subprocess.run([sys.executable,'-m','pip','install','-r',str(req),'--disable-pip-version-check'],check=True)
import torch, transformers, streamlit
print('Python:', sys.version.split()[0], '| torch:', torch.__version__, '| transformers:',transformers.__version__, '| Streamlit:',streamlit.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (CPU training will be slow)')
if int(transformers.__version__.split('.')[0]) >= 5:
    raise RuntimeError('Transformers 5 is not supported by this project. Reinstall with transformers>=4.44,<5 then restart runtime.')


## 3) فحص إلزامي لملفات التدريب واتساق الوسوم

التوقف عند غياب الملفات أفضل من تدريب ناقص أو توليد نتائج مضللة.

In [ ]:
from src.common import CONFIG, DATA_DIR, MODELS_DIR, REPORTS_DIR
from src.data.loaders import load_classification, load_ner
import pandas as pd
for task in ('opportunity_type','geographic_scope'):
    splits, lm = load_classification(task,DATA_DIR,splits=['train','validation','test'])
    if not splits['train'].shape[0] or not splits['validation'].shape[0] or not splits['test'].shape[0]:
        raise ValueError(f'{task}: empty train, validation, or test split')
    labels = lm['label2id']
    print(task, 'train',len(splits['train']),'val',len(splits['validation']),'test',len(splits['test']), 'labels',len(labels))
ner, bio = load_ner(DATA_DIR,splits=['train','validation','test'])
if not all(len(ner[s]) for s in ('train','validation','test')):
    raise ValueError('NER has an empty split')
print('NER:',{s:len(ner[s]) for s in ner}, '| BIO tags:', len(bio))
if not (DATA_DIR/'unique_opportunities.csv').is_file():
    raise FileNotFoundError('unique_opportunities.csv is required for the Streamlit pages')
print('Preflight: PASS — files and labels structurally consistent; this is NOT a semantic leakage or Gold audit')
del ner, splits, lm
import gc
gc.collect()


## 4) إصلاح AMP/Scheduler في كود التدريب — مع نسخ احتياطية

In [ ]:
# AMP scheduler fix: applies to local training code and the persistent Drive source.
#!/usr/bin/env python3
"""Patch incorrect LR scheduler stepping when AMP skips optimizer updates.

Safe/idempotent: backs up changed Python modules, validates syntax before writing.
Both a fast local Colab worktree and a persistent Drive tree may be patched.
This does NOT alter existing weights or claim to retrain models.
"""
from __future__ import annotations

import argparse
import ast
import datetime as dt
import os
import re
from pathlib import Path

MARKER = "AMP_SAFE_SCHEDULER_STEP_V1"
FILES = ("src/training/train_classifier.py", "src/training/train_ner.py")
PATTERN = re.compile(
    r"^(?P<indent>[ \t]*)scaler\.step\(opt\)\r?\n"
    r"(?P=indent)scaler\.update\(\)\r?\n"
    r"(?P=indent)opt\.zero_grad\((?:set_to_none=True)?\)\r?\n"
    r"(?P=indent)sch\.step\(\)", re.MULTILINE,
)


def patch_text(source: str, name: str) -> tuple[str, bool]:
    if MARKER in source:
        return source, False
    matches = list(PATTERN.finditer(source))
    if len(matches) != 1:
        raise RuntimeError(f"{name}: expected exactly one optimizer/scheduler block, found {len(matches)}; not changing file")

    indent = matches[0].group("indent")
    replacement = "\n".join((
        f"{indent}# {MARKER}: schedule only after a real optimizer update",
        f"{indent}scale_before_update = scaler.get_scale()",
        f"{indent}scaler.step(opt)",
        f"{indent}scaler.update()",
        f"{indent}step_skipped_by_amp = bool(amp and scaler.get_scale() < scale_before_update)",
        f"{indent}opt.zero_grad(set_to_none=True)",
        f"{indent}if not step_skipped_by_amp:",
        f"{indent}    optimizer_updates += 1",
        f"{indent}    sch.step()",
        f"{indent}else:",
        f"{indent}    amp_skipped_updates += 1",
    ))
    source = source[:matches[0].start()] + replacement + source[matches[0].end():]

    # Counters reset once per epoch, before the first minibatch is processed.
    epoch_counter = re.compile(r"^(?P<indent>[ \t]*)tot, n = 0\.0, 0[ \t]*$", re.MULTILINE)
    found = list(epoch_counter.finditer(source))
    if len(found) != 1:
        raise RuntimeError(f"{name}: no unique epoch counter initialization; not changing file")
    m = found[0]
    new_counter = m.group(0) + "\n" + m.group("indent") + "optimizer_updates = 0\n" + m.group("indent") + "amp_skipped_updates = 0"
    source = source[:m.start()] + new_counter + source[m.end():]

    # Save AMP diagnostics alongside validation metrics in training_history.csv.
    target = "        hist.append(row)"
    if source.count(target) != 1:
        raise RuntimeError(f"{name}: could not add AMP diagnostics to history")
    source = source.replace(target,
        "        row['optimizer_updates'] = optimizer_updates\n"
        "        row['amp_skipped_updates'] = amp_skipped_updates\n"
        "        hist.append(row)", 1)

    # Do not silently accept an epoch in which no weights could be updated.
    if name == "train_classifier.py":
        entry = "        lg = predict(model, dl_va, dev, amp)"
    else:
        entry = "        model.eval()"
    if source.count(entry) != 1:
        raise RuntimeError(f"{name}: could not insert update-validation guard")
    source = source.replace(entry,
        "        if optimizer_updates == 0:\n"
        "            raise RuntimeError('AMP skipped every optimizer step in this epoch; investigate nonfinite gradients.')\n"
        "        if amp_skipped_updates:\n"
        "            print(f'[AMP] epoch {ep}: skipped {amp_skipped_updates} nonfinite-gradient update(s)', flush=True)\n"
        + entry, 1)
    ast.parse(source, filename=name)
    return source, True


def patch_project(project_root: Path) -> None:
    if not (project_root / "src" / "common.py").exists():
        raise FileNotFoundError(f"Project not found: {project_root}")
    prepared = []
    for rel in FILES:
        p = project_root / rel
        if not p.is_file():
            raise FileNotFoundError(f"Required training source missing: {p}")
        raw = p.read_text(encoding="utf-8")
        edited, changed = patch_text(raw, p.name)
        prepared.append((p, raw, edited, changed))
    # All files validated before any changes are committed.
    for p, original, edited, changed in prepared:
        if not changed:
            print("ALREADY_PATCHED:", p)
            continue
        suffix = dt.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        bak = p.with_name(p.name + ".before_amp_fix_" + suffix + ".bak")
        bak.write_text(original, encoding="utf-8")
        tmp = p.with_name(p.name + ".tmp_amp_fix")
        tmp.write_text(edited, encoding="utf-8")
        os.replace(tmp, p)
        print("PATCHED:", p)
        print("BACKUP:", bak)
    print("SUCCESS: AMP scheduler is guarded; Python files syntax-checked. Models and datasets untouched.")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--project-root", default="/content/lmi")
    ap.add_argument("--drive-project", help="Optionally patch persistent project source in Google Drive")
    args = ap.parse_args()
    roots = [Path(args.project_root)]
    if args.drive_project:
        roots.append(Path(args.drive_project))
    for root in dict.fromkeys(roots):
        patch_project(root)



patch_project(Path(ROOT))
patch_project(Path(PROJ))


## 5) فحص وإصلاح خطأ البحث ذي النتائج الصفرية

يعالج فقدان الأعمدة عند عدم وجود نتائج بحث (ومنها `posting_date`) ويُثبت التصحيح في Drive أيضاً. التعديل محدود في دالة البحث ويتوقف عند اختلاف بنية الكود.

In [ ]:
import ast, datetime
SEARCH_MARKER = '.astype(bool)]'

def fix_search_result_masks(project):
    target=Path(project)/'src/integration/engine.py'
    code=target.read_text(encoding='utf-8')
    start=code.find('    def search(self,')
    end=code.find('    def governorates(',start)
    if start<0 or end<0:
        raise RuntimeError('Market.search boundary not found; refusing unsafe change')
    portion=code[start:end]
    old_new=[
        ('u = u[u.specializations.map(lambda s: specialization in split_multi(s))]',
         'u = u.loc[u.specializations.map(lambda s: specialization in split_multi(s)).astype(bool)]'),
        ('u = u[hay.map(lambda h: all(t in h for t in terms))]',
         'u = u.loc[hay.map(lambda h: all(t in h for t in terms)).astype(bool)]')]
    for old,new in old_new:
        if old in portion: portion=portion.replace(old,new,1)
        elif new not in portion: raise RuntimeError('Unexpected search code; not patching '+str(target))
    edited=code[:start]+portion+code[end:]
    ast.parse(edited)
    if edited != code:
        suffix=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
        (target.parent/(target.name+'.before_search_'+suffix+'.bak')).write_text(code,encoding='utf-8')
        target.write_text(edited,encoding='utf-8')
        print('PATCHED:',target)
    else:
        print('ALREADY PATCHED:',target)

for project in (ROOT,PROJ):
    fix_search_result_masks(project)
check_search = """from src.integration.engine import Market
m=Market(); a=m.search('qzxw1998nonexistent',limit=3)
assert len(a)==0 and 'posting_date' in a.columns
b=m.search('',limit=3); assert not b.empty and 'posting_date' in b.columns
print('Search: PASS (empty and populated results)')"""
subprocess.run([sys.executable,'-c',check_search],cwd=ROOT,env=os.environ.copy(),check=True)


## 6) تدريب AraBERT لتصنيف نوع الفرصة والنطاق الجغرافي

الدفتر لا يعتمد على وجود ملف `training_config.json` وحده: يتحقق أيضاً من تحميل الأوزان وTokenizer. إذا كان أحدهما تالفاً، يتوقف برسالة بدلاً من تجاوزه. **المهمة التي انقطع تدريبها قبل اكتمالها ستبدأ من الصفر** وفق كود التدريب الحالي.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification, AutoTokenizer
import gc

def verified_training(d, task):
    d=Path(d)
    metadata=d/'training_config.json'
    config=d/'config.json'
    if not metadata.is_file() or not config.is_file():
        return False
    meta=json.loads(metadata.read_text(encoding='utf-8'))
    if meta.get('task')!=task:
        raise ValueError(f'Unexpected training task in {metadata}: {meta.get("task")!r}')
    try:
        tok=AutoTokenizer.from_pretrained(d,local_files_only=True)
        cls = AutoModelForTokenClassification if task=='ner' else AutoModelForSequenceClassification
        mdl=cls.from_pretrained(d,local_files_only=True)
        if mdl.config.num_labels<=1:
            raise ValueError('Model has fewer than two output labels')
        if len(tok) > mdl.get_input_embeddings().weight.shape[0]:
            raise ValueError('Tokenizer vocabulary does not match model embeddings')
        if task != 'ner':
            expected=json.loads((DATA_DIR / f'classification_{task}/label_map.json').read_text(encoding='utf-8'))['id2label']
            actual={str(k):v for k,v in mdl.config.id2label.items()}
            if actual!=expected:
                raise ValueError('Checkpoint id2label does not match current training dataset label_map')
        else:
            tags=json.loads((DATA_DIR/'ner/label_list.json').read_text(encoding='utf-8'))['bio_tags']
            if [mdl.config.id2label[i] for i in range(len(tags))]!=tags:
                raise ValueError('NER checkpoint tags do not match dataset BIO tag list')
        del mdl,tok
        gc.collect()
    except Exception as exc:
        raise RuntimeError(f'Checkpoint metadata exists but model is incomplete/corrupted: {d}. Do not mark it trained. Error: {exc}') from exc
    return True

M=Path(os.environ['LMI_MODELS_DIR'])
for task in ('opportunity_type','geographic_scope'):
    d=M/'classification'/task
    if verified_training(d,task):
        print('VALID TRAINED CHECKPOINT — SKIP:',task)
        continue
    print('TRAINING:',task,flush=True)
    t=time.time()
    subprocess.run([sys.executable,'-m','src.training.train_classifier','--task',task],cwd=ROOT,env=os.environ.copy(),check=True)
    if not verified_training(d,task):
        raise RuntimeError(f'Classifier ended without loadable completion checkpoint: {task}')
    print('COMPLETED:',task,'minutes',round((time.time()-t)/60,1))


## 7) تدريب نموذج NER باستخدام AraBERT

In [ ]:
d=M/'ner'
if verified_training(d,'ner'):
    print('VALID TRAINED CHECKPOINT — SKIP: NER')
else:
    print('TRAINING: NER',flush=True)
    t=time.time()
    subprocess.run([sys.executable,'-m','src.training.train_ner'],cwd=ROOT,env=os.environ.copy(),check=True)
    if not verified_training(d,'ner'):
        raise RuntimeError('NER ended without a complete loadable checkpoint')
    print('COMPLETED: NER, minutes',round((time.time()-t)/60,1))


## 8) تقييم Test الفعلي — لا نستخدم تقارير قديمة تلقائياً

يُعاد تنفيذ كل تقييم حتى لو كان `metrics.json` موجوداً، لأن وجود الملف ليس دليلاً على ارتباطه بالأوزان الجديدة. تُحفظ النتائج في Drive.

In [ ]:
for task in ('opportunity_type','geographic_scope'):
    if not verified_training(M/'classification'/task,task):
        raise RuntimeError(f'{task} was not trained')
    subprocess.run([sys.executable,'-m','src.evaluation.evaluate_classifier','--task',task],cwd=ROOT,env=os.environ.copy(),check=True)
if not verified_training(M/'ner','ner'):
    raise RuntimeError('NER was not trained')
subprocess.run([sys.executable,'-m','src.evaluation.evaluate_ner'],cwd=ROOT,env=os.environ.copy(),check=True)
subprocess.run([sys.executable,'scripts/build_results_section.py'],cwd=ROOT,env=os.environ.copy(),check=True)
(PROJ / 'docs').mkdir(parents=True,exist_ok=True)
shutil.copy2(ROOT/'docs/RESULTS.md',PROJ/'docs/RESULTS.md')
from IPython.display import Markdown,display
display(Markdown((PROJ/'docs/RESULTS.md').read_text(encoding='utf-8')))


## 9) الاختبارات الآلية — إظهار الخطأ الحقيقي وعدم إخفائه

لا نستخدم `| tail` لأن ذلك قد يجعل خلية الاختبارات تظهر ناجحة رغم فشل Pytest.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_dl_system.py','-p','no:cacheprovider'],cwd=ROOT,env=os.environ.copy(),check=True)
print('Pytest: PASS')


## 10) (اختياري) تصدير النماذج والتقارير إلى Drive

هذه الخلية مستقلة عن خلية تشغيل التطبيق، ويمكن تشغيلها وحدها بعد نجاح التدريب. **أوزان fp16** توفر المساحة لكنها تختلف رقمياً قليلاً عن أوزان fp32؛ لذلك يُصدَّر بدون تحويل افتراضياً.

In [ ]:
MAKE_FP16_EXPORT=False
args=[sys.executable,'scripts/package_results.py','--out',str(MY)]
if MAKE_FP16_EXPORT:
    args.append('--fp16')
subprocess.run(args,cwd=ROOT,env=os.environ.copy(),check=True)
for name in ('lmi_models.zip','lmi_reports.zip'):
    f=MY/name
    if not f.is_file() or f.stat().st_size==0:
        raise RuntimeError(f'Export missing or empty: {f}')
    print(name,'size MB:',round(f.stat().st_size/1e6,1))


## 11) تشغيل Streamlit وفحص استجابة الخادم قبل مشاركة الرابط

الرابط المؤقت عام: لا تدخل بيانات شخصية أو سرية فيه. تحفظ سجلات الخادم للمراجعة في `/content/lmi_streamlit.log`. أوقف الخلية لإيقاف التطبيق.

In [ ]:
import urllib.request, re, threading, queue
cloud_bin=Path('/content/cloudflared')
if not cloud_bin.exists():
    url='https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(url,cloud_bin)
    cloud_bin.chmod(0o755)
logpath=Path('/content/lmi_streamlit.log')
app_log=logpath.open('w',encoding='utf-8')
app=subprocess.Popen([sys.executable,'-m','streamlit','run','app_dl/main.py','--server.port','8501','--server.headless','true'],
                     cwd=ROOT,env=os.environ.copy(),stdout=app_log,stderr=subprocess.STDOUT)
try:
    status=None
    for _ in range(40):
        if app.poll() is not None:
            raise RuntimeError('Streamlit exited; check /content/lmi_streamlit.log:\n'+logpath.read_text(encoding='utf-8')[-3500:])
        try:
            with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health',timeout=1) as response:
                if response.status==200:
                    status=200;break
        except Exception:
            time.sleep(1)
    if status!=200:
        raise RuntimeError('Streamlit did not respond; inspect /content/lmi_streamlit.log')
    print('Streamlit HTTP health: PASS')
    tun=subprocess.Popen([str(cloud_bin),'tunnel','--url','http://localhost:8501'],
                         stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    lines=queue.Queue()
    def reader():
        for line in tun.stdout:
            lines.put(line)
    threading.Thread(target=reader,daemon=True).start()
    public=None; deadline=time.monotonic()+75
    while time.monotonic()<deadline:
        if tun.poll() is not None and lines.empty():
            raise RuntimeError('cloudflared terminated before providing a link')
        try:
            line=lines.get(timeout=1)
        except queue.Empty:
            continue
        match=re.search(r'https://[\w-]+\.trycloudflare\.com',line)
        if match:
            public=match.group(0);break
    if not public:
        raise RuntimeError('No Cloudflare URL within 75s; inspect internet connectivity and tunnel service')
    print('App link:',public)
    while app.poll() is None and tun.poll() is None:
        time.sleep(5)
finally:
    for proc in (locals().get('tun'),app):
        if proc is not None and proc.poll() is None:
            proc.terminate()
    app_log.close()
    print('Streamlit log:',logpath)
